# 8. 데이터 증강과 일반화

이 노트북은 `07_GradCAM_시각화.ipynb` 다음 단계로, **비전 모델이 보지 못한 데이터에도 잘 동작하도록 만드는 핵심 전략** 인 데이터 증강(data augmentation)을 실습합니다.

앞선 노트북에서는 CNN 구조, ResNet 계열, transfer learning, Grad-CAM까지 따라오면서 모델이 어떻게 학습되고 무엇을 보는지 확인했습니다. 이제는 한 단계 더 나아가, **같은 모델이라도 학습 데이터를 어떻게 보여 주느냐에 따라 일반화 성능이 얼마나 달라지는지** 살펴보겠습니다.

이번 노트북의 목표는 다음과 같습니다.

- 데이터 증강이 왜 필요한지 이해합니다.
- `RandomHorizontalFlip`, `RandomCrop`, `ColorJitter` 같은 기본 증강을 적용해 봅니다.
- 같은 모델을 `무증강`과 `증강 적용` 두 조건으로 학습해 비교합니다.
- train accuracy와 validation/test accuracy 차이를 통해 일반화를 해석합니다.


## 8-1. 왜 데이터 증강이 필요할까?

비전 모델은 학습 데이터에 자주 등장하는 패턴을 빠르게 익힙니다. 그런데 데이터가 충분히 다양하지 않으면, 모델은 물체의 본질보다 **배경, 위치, 색감, 구도 같은 우연한 단서** 에 지나치게 의존할 수 있습니다.

데이터 증강은 원본 이미지를 조금씩 변형해, 모델이 더 다양한 상황을 보게 만드는 방법입니다.

- 좌우 반전
- 약간의 이동과 crop
- 밝기와 대비 변화
- 작은 회전이나 크기 변화

이 과정을 통해 모델은 특정 한 장면을 외우기보다, **조금 달라져도 유지되는 공통 특징** 을 학습하게 됩니다. 이것이 일반화(generalization)에 직접 연결됩니다.


## 8-2. 준비

이번 실험은 CIFAR-10과 작은 CNN 또는 ResNet 스타일 모델로도 충분히 확인할 수 있습니다. 계산 부담을 너무 키우지 않기 위해, 여기서는 CIFAR-10 크기에 맞는 비교적 가벼운 ResNet 스타일 모델을 사용합니다.

핵심은 모델 자체보다도 **같은 모델을 어떤 transform으로 학습시키느냐** 입니다.


In [ ]:
# 필요 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)


## 8-3. 무증강 데이터와 증강 데이터 준비

여기서는 학습용 transform만 다르게 두고, validation/test는 항상 같은 평가 기준으로 유지합니다. 이렇게 해야 증강의 효과를 공정하게 비교할 수 있습니다.


In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

plain_train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

aug_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

plain_train_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=plain_train_transform)
aug_train_full = datasets.CIFAR10(root='./data', train=True, download=False, transform=aug_train_transform)
train_eval_full = datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

classes = plain_train_full.classes
train_size = 45000
val_size = 5000

generator = torch.Generator().manual_seed(42)
plain_train_dataset, _ = random_split(plain_train_full, [train_size, val_size], generator=generator)

generator = torch.Generator().manual_seed(42)
aug_train_dataset, _ = random_split(aug_train_full, [train_size, val_size], generator=generator)

generator = torch.Generator().manual_seed(42)
_, val_dataset = random_split(train_eval_full, [train_size, val_size], generator=generator)

plain_train_loader = DataLoader(plain_train_dataset, batch_size=128, shuffle=True)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print('Classes:', classes)
print('Plain train samples:', len(plain_train_dataset))
print('Aug train samples  :', len(aug_train_dataset))
print('Validation samples :', len(val_dataset))
print('Test samples       :', len(test_dataset))


In [ ]:
def denormalize(image):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (image.cpu() * std_tensor + mean_tensor).clamp(0, 1)


images, labels = next(iter(aug_train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


위 이미지를 보면 같은 CIFAR-10이라도 위치와 색감이 조금씩 달라집니다. 이런 변화는 사람이 보기에는 여전히 같은 클래스이지만, 모델 입장에서는 더 다양한 학습 사례를 보게 되는 셈입니다.


## 8-4. 비교에 사용할 작은 ResNet 스타일 모델

이번 실험은 증강 효과를 보는 것이 목적이므로, 너무 무거운 모델 대신 비교적 가벼운 residual network를 사용합니다. 중요한 점은 **무증강 실험과 증강 실험에서 모델 구조를 완전히 동일하게 유지** 하는 것입니다.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.layers = nn.Sequential(
            ResidualBlock(32, 32, stride=1),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64, stride=1),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128, stride=1)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


## 8-5. 공통 학습 함수

무증강과 증강을 공정하게 비교하려면 optimizer, epoch 수, learning rate는 같게 두고 오직 train transform만 바꿔야 합니다.


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = []

    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append((train_loss, train_acc, val_loss, val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return model, history, criterion


## 8-6. 실험 1: 무증강으로 학습

먼저 원본 이미지 그대로만 사용해 학습합니다. 보통 이 경우 train accuracy는 빠르게 오르지만, validation/test 성능은 그만큼 따라오지 못할 수 있습니다.


In [ ]:
epochs = 5
learning_rate = 0.001

plain_model = SmallResNet(num_classes=len(classes))
print('=== 무증강 학습 ===')
plain_model, plain_history, plain_criterion = train_model(
    plain_model,
    plain_train_loader,
    val_loader,
    epochs=epochs,
    lr=learning_rate
)
plain_test_loss, plain_test_acc = evaluate(plain_model, test_loader, plain_criterion)
print(f'무증강 Test Loss: {plain_test_loss:.4f} | Test Acc: {plain_test_acc:.4f}')


## 8-7. 실험 2: 증강을 적용해 학습

이번에는 모델 구조와 optimizer는 그대로 두고, 학습용 이미지에만 augmentation을 적용합니다. 일반적으로는 train accuracy가 조금 덜 빠르게 오르더라도, validation/test 성능이 더 안정적일 수 있습니다.


In [ ]:
aug_model = SmallResNet(num_classes=len(classes))
print('=== 증강 적용 학습 ===')
aug_model, aug_history, aug_criterion = train_model(
    aug_model,
    aug_train_loader,
    val_loader,
    epochs=epochs,
    lr=learning_rate
)
aug_test_loss, aug_test_acc = evaluate(aug_model, test_loader, aug_criterion)
print(f'증강 적용 Test Loss: {aug_test_loss:.4f} | Test Acc: {aug_test_acc:.4f}')


## 8-8. 학습 곡선 비교

이제 두 실험의 loss와 accuracy를 함께 비교합니다. 핵심은 단순히 train accuracy가 높은 쪽이 아니라, **validation/test까지 얼마나 안정적으로 따라오는가** 입니다.


In [ ]:
epochs_axis = range(1, len(plain_history) + 1)

plain_train_losses = [item[0] for item in plain_history]
plain_train_accs = [item[1] for item in plain_history]
plain_val_losses = [item[2] for item in plain_history]
plain_val_accs = [item[3] for item in plain_history]

aug_train_losses = [item[0] for item in aug_history]
aug_train_accs = [item[1] for item in aug_history]
aug_val_losses = [item[2] for item in aug_history]
aug_val_accs = [item[3] for item in aug_history]

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs_axis, plain_train_losses, marker='o', label='Plain Train Loss')
plt.plot(epochs_axis, plain_val_losses, marker='o', label='Plain Val Loss')
plt.plot(epochs_axis, aug_train_losses, marker='s', label='Aug Train Loss')
plt.plot(epochs_axis, aug_val_losses, marker='s', label='Aug Val Loss')
plt.title('Loss 비교')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs_axis, plain_train_accs, marker='o', label='Plain Train Acc')
plt.plot(epochs_axis, plain_val_accs, marker='o', label='Plain Val Acc')
plt.plot(epochs_axis, aug_train_accs, marker='s', label='Aug Train Acc')
plt.plot(epochs_axis, aug_val_accs, marker='s', label='Aug Val Acc')
plt.title('Accuracy 비교')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 3, 3)
plt.bar(['Plain', 'Augmentation'], [plain_test_acc, aug_test_acc], color=['#94a3b8', '#2563eb'])
plt.ylim(0, 1)
plt.title('Test Accuracy')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()


보통 augmentation을 쓰면 train accuracy는 조금 낮게 나와도, validation/test에서는 더 좋은 결과를 보이는 경우가 많습니다. 이것은 모델이 학습 데이터를 덜 외우고, 더 일반적인 특징을 배우고 있다는 신호일 수 있습니다.


## 8-9. 과적합 관점에서 해석하기

일반화는 결국 과적합과 연결됩니다. 아래처럼 해석하면 좋습니다.

- `train accuracy`는 계속 높아지는데 `val accuracy`가 정체되면 과적합 가능성이 큽니다.
- augmentation은 학습 데이터를 더 어렵고 다양하게 만들어, 이런 과적합을 늦추는 역할을 합니다.
- 따라서 augmentation은 단순한 이미지 장난이 아니라, **일반화 성능을 높이기 위한 규제(regularization) 전략** 으로 볼 수 있습니다.


In [ ]:
generalization_gap_plain = plain_train_accs[-1] - plain_val_accs[-1]
generalization_gap_aug = aug_train_accs[-1] - aug_val_accs[-1]

print(f'무증강 일반화 격차  : {generalization_gap_plain:.4f}')
print(f'증강 적용 일반화 격차: {generalization_gap_aug:.4f}')


일반화 격차(generalization gap)가 더 작다면, 적어도 그 시점에서는 train 성능이 validation 성능에 비해 덜 과하게 벌어졌다고 해석할 수 있습니다. 물론 최종 판단은 validation/test 성능과 함께 봐야 합니다.


## 8-10. 예측 결과 확인

마지막으로 augmentation을 사용한 모델이 실제 테스트 이미지에서 어떤 예측을 하는지 직접 확인합니다.


In [ ]:
aug_model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = aug_model(images)
    preds = outputs.argmax(dim=1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label, pred in zip(axes.flat, images[:8], labels[:8], preds[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(f'T: {classes[label]}\nP: {classes[pred]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 8-11. 추가 실험 아이디어

이번 노트북을 바탕으로 다음 실험을 이어서 해 볼 수 있습니다.

- `RandomErasing`이나 `AutoAugment` 추가하기
- augmentation 강도를 너무 크게 했을 때 성능이 오히려 떨어지는지 보기
- 같은 실험을 transfer learning 모델에 적용해 보기
- `CutMix`, `MixUp` 같은 더 강한 regularization 기법으로 확장하기


## 정리

이번 노트북의 핵심은 다음과 같습니다.

- 데이터 증강은 학습 이미지를 다양하게 변형해 모델이 더 일반적인 특징을 배우도록 돕습니다.
- 무증강 학습은 train accuracy가 빠르게 오르더라도 validation/test 성능이 덜 따라올 수 있습니다.
- augmentation은 과적합을 줄이고 일반화 성능을 높이는 대표적인 전략입니다.
- 비전 모델을 더 잘 학습시키는 다음 단계에서는, 모델 구조만큼이나 데이터와 학습 전략이 중요합니다.

다음 단계로는 scheduler, weight decay, early stopping까지 포함한 **학습 전략 고도화** 로 이어가면 자연스럽습니다.
